In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Open Orders via the real sidebar (label verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Orders')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='Sales Orders']")))
    print("PASS: Orders page opened")

    # Real content markers (verified in OrdersPage.jsx)
    body = driver.find_element(By.TAG_NAME, "body").text
    assert "Recorded sales" in body and "Invoice" in body
    search = driver.find_element(By.XPATH, "//input[@aria-label='Search orders']")
    assert search.is_displayed()
    assert "Orders" in body and "Revenue" in body and "Line Items" in body
    print("PASS: Orders content verified")

    # Detail view: rows are read-only in the current frontend (no onClick on tr.pos-row)
    rows = [r for r in driver.find_elements(By.XPATH, "//tbody[contains(@class, 'pos-tbody-divided')]//tr[contains(@class, 'pos-row')]") if r.is_displayed()]
    if rows:
        print("First order:", rows[0].text.split("\n")[0])
        print("SKIP: No existing orders available for detail testing (rows are read-only, no detail view in current frontend)")
    else:
        assert "No orders found" in body, "Orders list is neither populated nor showing its empty state."
        print("Empty state shown: No orders found")
        print("SKIP: No existing orders available for detail testing")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("35_orders_FAIL.png")
finally:
    driver.quit()